# 22A — ConvNeXt asset diagnostic (CPU only)

Attach only the exact GoodPJW `rsna-knee-2-5d-convnext-reader` Dataset used by notebook 22. Set Accelerator **None**, Internet **Off**, then Save & Run All. This notebook does not import torch, run inference, or submit. Download its Output ZIP and send `convnext_asset_diagnostic.json`.


In [ ]:
from pathlib import Path
import hashlib, json

BASE = Path('/kaggle/input')
NAME = 'rsna-knee-2-5d-convnext-reader'
EXPECTED = {
    'cnxt_v0_fold0.pt': 'abd3ffbd483e31d77eb6212a2f295ee791203addb73bd6844b1bd34290a10e2f',
    'cnxt_v0_fold1.pt': 'f3d42e4e83c3b06187d04b5b5886b1ee05f9ec80d30eb66da9b05d3fd53b1aa3',
    'cnxt_v0_fold2.pt': 'eb3d9d7e11315bcb47b55f408d19262fbc72f8bf76a6af3d94c73ba8a95f1010',
    'preprocess.py': '68601ad84ec3a0f030b5c2d37cc0d657cd3e12b17f7130e7385fba9648d324b6',
    'knee.py': '314c2af387a5cbd1d25f192f19f4ecc6d4ae4b30403e92e9d9b3ed2c325880b1',
    'infer.py': '39d4517cc33c4bb1e4a03cfb49a0673a8789055de8c3cc0957c6e3035f6be4af',
}
roots = [BASE/'datasets'/'goodpjw2008'/NAME, BASE/NAME]
out = {'dataset_slug': 'goodpjw2008/'+NAME, 'expected_sha256': EXPECTED,
       'top_level_inputs': sorted(p.name for p in BASE.iterdir()) if BASE.is_dir() else [],
       'mounts': []}
for root in roots:
    item = {'root': str(root), 'exists': root.is_dir(), 'files': [], 'assets': []}
    if root.is_dir():
        for p in sorted(root.rglob('*')):
            if not p.is_file() or len(p.relative_to(root).parts) > 5:
                continue
            rel = str(p.relative_to(root))
            if len(item['files']) < 60:
                item['files'].append(rel)
            if p.name in EXPECTED:
                with p.open('rb') as handle:
                    digest = hashlib.file_digest(handle, 'sha256').hexdigest()
                item['assets'].append({'relative_path': rel, 'bytes': p.stat().st_size,
                    'sha256': digest, 'matches_expected': digest == EXPECTED[p.name]})
    out['mounts'].append(item)
Path('/kaggle/working/convnext_asset_diagnostic.json').write_text(json.dumps(out, indent=2)+'\n')
print(json.dumps(out, indent=2))
